# ViT APJN experiments notebook

This notebook follows the `vit-dyt` bootstrap flow from the reference Colab before running the APJN experiments.

In [ ]:
# Optional: install TeX/font packages for publication-style plots.
# Run this cell only if you want the LaTeX/NeurIPS-like font setup below.
!apt-get -qq update
!apt-get -qq install texlive-latex-extra texlive-fonts-recommended dvipng cm-super

In [ ]:
# Optional: configure NeurIPS-like LaTeX fonts.
# Run after the apt-get cell above.
# If you want the simpler Times-like fallback instead, use `configure_times_like_tex_fonts()`.
from vit_apjn_notebook_helpers import configure_neurips_like_tex_fonts, configure_times_like_tex_fonts

configure_times_like_tex_fonts()

In [ ]:
# Colab/local setup for vit-dyt.
from pathlib import Path
import subprocess
import sys

REPO_DIR = Path('vit-dyt')
INSTALL_TIMM = True
INSTALL_REQUIREMENTS = True
CLONE_IF_MISSING = True

if INSTALL_TIMM:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'timm'], check=True)

if not REPO_DIR.exists() and CLONE_IF_MISSING:
    subprocess.run(['git', 'clone', 'https://github.com/labofdoubt/vit-dyt', str(REPO_DIR)], check=True)

REQ = REPO_DIR / 'requirements.txt'
if INSTALL_REQUIREMENTS and REQ.exists():
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', str(REQ)], check=True)

if REPO_DIR.exists() and str(REPO_DIR.resolve()) not in sys.path:
    sys.path.insert(0, str(REPO_DIR.resolve()))

print('Repo directory:', REPO_DIR.resolve() if REPO_DIR.exists() else 'missing')

Repo directory: /content/vit-dyt


In [ ]:
# Common imports and helper bootstrap.
%matplotlib inline

import numpy as np
import matplotlib.pyplot as plt

from vit_apjn_notebook_helpers import *

In [ ]:
bootstrap_vit_dyt_repo(
    REPO_DIR,
    clone_if_missing=CLONE_IF_MISSING,
    install_requirements=INSTALL_REQUIREMENTS,
    install_timm=INSTALL_TIMM,
)
set_pub_style()
print('DEVICE =', DEVICE)

DEVICE = cuda


In [ ]:
# Global defaults and top-level tuning handles.
DEFAULT_DEPTH = 64
DEFAULT_ALPHAS = tuple(np.arange(0.1, 1.9 + 1e-9, 0.2).astype(float))
DEFAULT_Q0_GRID = tuple(np.linspace(0.0, 2.0, 41))
DEFAULT_P0_GRID = tuple(np.linspace(0.0, 2.0, 41))
DEFAULT_PRELN_SCALE_GRID = tuple(np.linspace(0.4, 8.0, 61))

ATTN_MULT = 1.0
MLP_MULT = 2.0

MODEL_CFG = ModelConfig(
    model_name='vit_base_patch16_224',
    depth=DEFAULT_DEPTH,
    num_classes=100,
    img_size=224,
    replace_gelu_with_relu=True,
    inplace_relu=False,
    seed=0,
)
MEAN_FIELD_CFG = build_mean_field_cfg_for_vit_base(
    depth=MODEL_CFG.depth,
    attn_mult=ATTN_MULT,
    mlp_mult=MLP_MULT
    )
STYLE_CFG = FinalThreePanelStyleConfig(
    title_fs=18,
    label_fs=18,
    annotation_fs=18,
    colorbar_pad=0.04,
    )

# Direct plotting overrides for notebook figures.
PLOT_TICK_FS = 16
PLOT_LABEL_FS = 18
PLOT_TITLE_FS = 18
PLOT_ANNOTATION_FS = 18
PLOT_ALPHA_LEGEND_FS = 16

EQ_PANEL_WIDTH_RATIOS = (1.0, 1.5, 1.5)
EQ_PANEL_GAP_AB = 0.18
EQ_PANEL_GAP_BC = 0.18

print('MODEL_CFG =', cfg_to_dict(MODEL_CFG))

MODEL_CFG = {'model_name': 'vit_base_patch16_224', 'depth': 64, 'num_classes': 100, 'img_size': 224, 'replace_gelu_with_relu': True, 'inplace_relu': False, 'seed': 0}


## 1. Equiangular p-trajectories and APJNs

In [ ]:
# Definitions.
EQ_Q0 = 1.0
EQ_P0 = 0.2
EQ_APJN_LAYERS = tuple(range(0, MODEL_CFG.depth + 1, 8))
EQ_DIRECT_LAYERS = tuple(range(0, MODEL_CFG.depth + 1, 8))

PERM_CFG_EQ = PermTokenConfig(
    batch_size=1,
    q0_init=EQ_Q0,
    p0_init=EQ_P0,
    attn_mult=ATTN_MULT,
    mlp_mult=MLP_MULT,
    alphas=DEFAULT_ALPHAS,
    perm_start_index=0,
    perm_n_replace=None,
    perm_seed=1235,
    perm_random_rotate=True,
    theory_n_tokens_override=None,
    num_model_inits=1,
)
APJN_CFG_EQ = APJNCifarConfig(
    input_source='equiangular',
    batch_size=1,
    equiangular_q0=EQ_Q0,
    equiangular_p0=EQ_P0,
    equiangular_seed=0,
    equiangular_random_rotate=True,
    apjn_layers=EQ_APJN_LAYERS,
    direct_layers=EQ_DIRECT_LAYERS,
    direct_source_block=0,
    alphas=DEFAULT_ALPHAS,
    j_num_draws=10,
    j_normalize_by='Y',
    num_model_inits=1,
    attn_mult=ATTN_MULT,
    mlp_mult=MLP_MULT,
)

In [ ]:
# Run.
perm_empirical_bundle_eq = run_perm_token_experiment(MODEL_CFG, PERM_CFG_EQ)
perm_theory_bundle_eq = compute_theory_qp_bundle(
    num_layers=MODEL_CFG.depth,
    alphas=DEFAULT_ALPHAS,
    n_tokens=perm_empirical_bundle_eq['n_tokens_ex_cls'],
    q0=EQ_Q0,
    p0=EQ_P0,
    mean_field_cfg=MEAN_FIELD_CFG,
)
eq_apjn_bundle = run_cifar_apjn_experiment(MODEL_CFG, APJN_CFG_EQ)

In [ ]:
# Plot.
# Tune panel widths with `EQ_PANEL_WIDTH_RATIOS=(wa, wb, wc)`.
# Tune panel gaps with `EQ_PANEL_GAP_AB` and `EQ_PANEL_GAP_BC`.
STYLE_CFG = FinalThreePanelStyleConfig(
    title_fs=20,
    label_fs=20,
    annotation_fs=20,
    colorbar_pad=0.12,
    )

fig_eq = plot_equangular_p_inverse_direct_figure(
    perm_empirical_bundle_eq,
    perm_theory_bundle_eq,
    eq_apjn_bundle,
    STYLE_CFG,
    panel_width_ratios=(1.0, 2.2, 2.2),
    panel_gap_ab=0.5,
    panel_gap_bc=0.4,
    tick_fs=PLOT_TICK_FS,
    label_fs=PLOT_LABEL_FS,
    alpha_legend_fs=PLOT_ALPHA_LEGEND_FS,
    title_fs=PLOT_TITLE_FS,
    annotation_fs=PLOT_ANNOTATION_FS,
)

## 2. Equiangular APJN sweep over $(\mathrm{MLP\_MULT}, \mathrm{ATTN\_MULT})$

In [ ]:
# Run the equiangular APJN/theory sweep once and store all bundles.
EQ_MULT_PAIRS = ((1.0, 1.0), (1.0, 2.0), (2.0, 1.0), (2.0, 2.0))
EQ_MULT_DISPLAY_ORDER = ((1.0, 2.0), (2.0, 2.0), (1.0, 1.0), (2.0, 1.0))

EQ_SWEEP_RESULTS = {}
for mlp_mult, attn_mult in EQ_MULT_PAIRS:
    mean_field_cfg = build_mean_field_cfg_for_vit_base(
        depth=MODEL_CFG.depth,
        attn_mult=attn_mult,
        mlp_mult=mlp_mult,
    )
    perm_cfg = PermTokenConfig(
        batch_size=1,
        q0_init=EQ_Q0,
        p0_init=EQ_P0,
        attn_mult=attn_mult,
        mlp_mult=mlp_mult,
        alphas=DEFAULT_ALPHAS,
        perm_start_index=0,
        perm_n_replace=None,
        perm_seed=1235,
        perm_random_rotate=True,
        theory_n_tokens_override=None,
        num_model_inits=1,
    )
    apjn_cfg = APJNCifarConfig(
        input_source='equiangular',
        batch_size=1,
        equiangular_q0=EQ_Q0,
        equiangular_p0=EQ_P0,
        equiangular_seed=0,
        equiangular_random_rotate=True,
        apjn_layers=EQ_APJN_LAYERS,
        direct_layers=EQ_DIRECT_LAYERS,
        direct_source_block=0,
        alphas=DEFAULT_ALPHAS,
        j_num_draws=10,
        j_normalize_by='Y',
        num_model_inits=1,
        attn_mult=attn_mult,
        mlp_mult=mlp_mult,
    )
    perm_empirical_bundle = run_perm_token_experiment(MODEL_CFG, perm_cfg)
    perm_theory_bundle = compute_theory_qp_bundle(
        num_layers=MODEL_CFG.depth,
        alphas=DEFAULT_ALPHAS,
        n_tokens=perm_empirical_bundle['n_tokens_ex_cls'],
        q0=EQ_Q0,
        p0=EQ_P0,
        mean_field_cfg=mean_field_cfg,
    )
    apjn_bundle = run_cifar_apjn_experiment(MODEL_CFG, apjn_cfg)
    EQ_SWEEP_RESULTS[(mlp_mult, attn_mult)] = {
        'mean_field_cfg': mean_field_cfg,
        'perm_cfg': perm_cfg,
        'apjn_cfg': apjn_cfg,
        'perm_empirical_bundle': perm_empirical_bundle,
        'perm_theory_bundle': perm_theory_bundle,
        'apjn_bundle': apjn_bundle,
    }
    print(f'Finished (MLP_MULT, ATTN_MULT)=({mlp_mult:.0f}, {attn_mult:.0f})')

In [ ]:
# Plot backward/direct APJN for the stored sweep bundles.
alpha_colors = _make_alpha_colors(DEFAULT_ALPHAS)
alpha_to_color = {float(alpha): alpha_colors[i] for i, alpha in enumerate(DEFAULT_ALPHAS)}

def _panel_title(mult_pair):
    mlp_mult, attn_mult = mult_pair
    return rf"$(\\mathrm{{MLP}}, \\mathrm{{ATTN}})=({int(mlp_mult)}, {int(attn_mult)})$"

def _extract_inverse_points_local(apjn_bundle):
    preln = {int(k): float(v) for k, v in apjn_bundle['preln_with_J']['J_raw'].items()}
    layers = sorted(preln.keys())
    derf = {}
    for entry in apjn_bundle['derf_pack_with_J']['results']:
        derf[float(entry['alpha'])] = {int(k): float(v) for k, v in entry['J_raw'].items()}
    return layers, preln, derf

def _extract_direct_points_local(apjn_bundle):
    source = int(apjn_bundle.get('direct_source_block', 0))
    preln = {
        source + int(k): float(np.mean(np.asarray(v, dtype=float)))
        for k, v in apjn_bundle['preln_with_J'].get('direct_J_raw', {}).items()
    }
    derf = {}
    for entry in apjn_bundle['derf_pack_with_J']['results']:
        derf[float(entry['alpha'])] = {
            source + int(k): float(np.mean(np.asarray(v, dtype=float)))
            for k, v in (entry.get('direct_J_raw') or {}).items()
        }
    layers = sorted(preln.keys()) if preln else sorted({k for vals in derf.values() for k in vals.keys()})
    return source, layers, preln, derf

fig = plt.figure(figsize=(18, 9.8))
outer = fig.add_gridspec(3, 4, height_ratios=(1.0, 1.0, 0.12), width_ratios=(1.0, 1.0, 1.0, 1.0), wspace=0.28, hspace=0.32)

axes_inv = []
axes_dir = []
for idx, mult_pair in enumerate(EQ_MULT_DISPLAY_ORDER):
    row, col = divmod(idx, 2)
    ax_inv = fig.add_subplot(outer[row, col])
    ax_dir = fig.add_subplot(outer[row, col + 2])
    axes_inv.append(ax_inv)
    axes_dir.append(ax_dir)

    result = EQ_SWEEP_RESULTS[mult_pair]
    perm_theory_bundle = result['perm_theory_bundle']
    apjn_bundle = result['apjn_bundle']
    l = np.asarray(perm_theory_bundle['l'], dtype=float)

    inv_layers, inv_preln, inv_derf = _extract_inverse_points_local(apjn_bundle)
    ax_inv.plot(l, perm_theory_bundle['preln']['J'], color='black', lw=2.0)
    if inv_layers:
        ax_inv.scatter(inv_layers, [inv_preln[k] for k in inv_layers], color='black', s=18, zorder=3)
    for alpha in DEFAULT_ALPHAS:
        alpha_key = float(alpha)
        color = alpha_to_color[alpha_key]
        ax_inv.plot(l, perm_theory_bundle['derf'][alpha_key]['J'], color=color, lw=1.8)
        if inv_layers:
            ax_inv.scatter(inv_layers, [inv_derf[alpha_key][k] for k in inv_layers], color=color, s=18, zorder=3)
    ax_inv.set_yscale('log')
    prettify_log_axis(ax_inv, 'y')
    prettify_axes(ax_inv)
    ax_inv.set_title(_panel_title(mult_pair), fontsize=PLOT_TITLE_FS)
    if row == 1:
        ax_inv.set_xlabel(r'$b$', fontsize=PLOT_LABEL_FS)
    if col == 0:
        ax_inv.set_ylabel(r'$\\mathcal{J}^{\\, B, b}$', fontsize=PLOT_LABEL_FS)
    ax_inv.tick_params(labelsize=PLOT_TICK_FS)

    direct_source, direct_layers, dir_preln, dir_derf = _extract_direct_points_local(apjn_bundle)
    ax_dir.plot(l, direct_curve_from_source(perm_theory_bundle['preln']['J_direct'], direct_source), color='black', lw=2.0)
    if direct_layers:
        ax_dir.scatter(direct_layers, [dir_preln[k] for k in direct_layers], color='black', s=18, zorder=3)
    for alpha in DEFAULT_ALPHAS:
        alpha_key = float(alpha)
        color = alpha_to_color[alpha_key]
        ax_dir.plot(l, direct_curve_from_source(perm_theory_bundle['derf'][alpha_key]['J_direct'], direct_source), color=color, lw=1.8)
        if direct_layers:
            ax_dir.scatter(direct_layers, [dir_derf[alpha_key][k] for k in direct_layers], color=color, s=18, zorder=3)
    ax_dir.set_yscale('log')
    prettify_log_axis(ax_dir, 'y')
    prettify_axes(ax_dir)
    ax_dir.set_title(_panel_title(mult_pair), fontsize=PLOT_TITLE_FS)
    if row == 1:
        ax_dir.set_xlabel(r'$b$', fontsize=PLOT_LABEL_FS)
    if col == 0:
        ax_dir.set_ylabel(r'$\\mathcal{J}^{\\, b, 0}$', fontsize=PLOT_LABEL_FS)
    ax_dir.tick_params(labelsize=PLOT_TICK_FS)

fig.text(0.25, 0.98, r'(a) backward APJN $\\mathcal{J}^{\\, B, b}$', ha='center', va='top', fontsize=PLOT_TITLE_FS + 2)
fig.text(0.75, 0.98, r'(b) direct APJN $\\mathcal{J}^{\\, b, 0}$', ha='center', va='top', fontsize=PLOT_TITLE_FS + 2)

cax = fig.add_subplot(outer[2, :])
center_shrink_axis(cax, width_scale=0.62, height_scale=0.75)
_draw_alpha_preln_legend_like_equangular(cax, np.asarray(DEFAULT_ALPHAS, dtype=float), alpha_colors, PLOT_ALPHA_LEGEND_FS)
fig.tight_layout(rect=(0.0, 0.03, 1.0, 0.95))
plt.show()

In [ ]:
# Plot theory K/J and K_direct/J_direct for the stored sweep bundles.
alpha_colors = _make_alpha_colors(DEFAULT_ALPHAS)
alpha_to_color = {float(alpha): alpha_colors[i] for i, alpha in enumerate(DEFAULT_ALPHAS)}

fig = plt.figure(figsize=(18, 9.8))
outer = fig.add_gridspec(3, 4, height_ratios=(1.0, 1.0, 0.12), width_ratios=(1.0, 1.0, 1.0, 1.0), wspace=0.28, hspace=0.32)

for idx, mult_pair in enumerate(EQ_MULT_DISPLAY_ORDER):
    row, col = divmod(idx, 2)
    ax_k = fig.add_subplot(outer[row, col])
    ax_k_direct = fig.add_subplot(outer[row, col + 2])

    perm_theory_bundle = EQ_SWEEP_RESULTS[mult_pair]['perm_theory_bundle']
    l = np.asarray(perm_theory_bundle['l'], dtype=float)

    preln_ratio = np.divide(
        perm_theory_bundle['preln']['K'],
        np.maximum(perm_theory_bundle['preln']['J'], 1e-300),
    )
    preln_direct_ratio = np.divide(
        perm_theory_bundle['preln']['K_direct'],
        np.maximum(perm_theory_bundle['preln']['J_direct'], 1e-300),
    )
    ax_k.plot(l, preln_ratio, color='black', lw=2.0)
    ax_k_direct.plot(l, preln_direct_ratio, color='black', lw=2.0)
    for alpha in DEFAULT_ALPHAS:
        alpha_key = float(alpha)
        color = alpha_to_color[alpha_key]
        derf_ratio = np.divide(
            perm_theory_bundle['derf'][alpha_key]['K'],
            np.maximum(perm_theory_bundle['derf'][alpha_key]['J'], 1e-300),
        )
        derf_direct_ratio = np.divide(
            perm_theory_bundle['derf'][alpha_key]['K_direct'],
            np.maximum(perm_theory_bundle['derf'][alpha_key]['J_direct'], 1e-300),
        )
        ax_k.plot(l, derf_ratio, color=color, lw=1.8)
        ax_k_direct.plot(l, derf_direct_ratio, color=color, lw=1.8)

    prettify_axes(ax_k)
    prettify_axes(ax_k_direct)
    ax_k.set_title(_panel_title(mult_pair), fontsize=PLOT_TITLE_FS)
    ax_k_direct.set_title(_panel_title(mult_pair), fontsize=PLOT_TITLE_FS)
    if row == 1:
        ax_k.set_xlabel(r'$l$', fontsize=PLOT_LABEL_FS)
        ax_k_direct.set_xlabel(r'$l$', fontsize=PLOT_LABEL_FS)
    if col == 0:
        ax_k.set_ylabel(r'$K[l] / J[l]$', fontsize=PLOT_LABEL_FS)
        ax_k_direct.set_ylabel(r'$K_{\\mathrm{direct}}[l] / J_{\\mathrm{direct}}[l]$', fontsize=PLOT_LABEL_FS)
    ax_k.tick_params(labelsize=PLOT_TICK_FS)
    ax_k_direct.tick_params(labelsize=PLOT_TICK_FS)

fig.text(0.25, 0.98, r'(a) theory $K[l] / J[l]$', ha='center', va='top', fontsize=PLOT_TITLE_FS + 2)
fig.text(0.75, 0.98, r'(b) theory $K_{\\mathrm{direct}}[l] / J_{\\mathrm{direct}}[l]$', ha='center', va='top', fontsize=PLOT_TITLE_FS + 2)

cax = fig.add_subplot(outer[2, :])
center_shrink_axis(cax, width_scale=0.62, height_scale=0.75)
_draw_alpha_preln_legend_like_equangular(cax, np.asarray(DEFAULT_ALPHAS, dtype=float), alpha_colors, PLOT_ALPHA_LEGEND_FS)
fig.tight_layout(rect=(0.0, 0.03, 1.0, 0.95))
plt.show()